# Lesson 11: Quantization Foundations & llama.cpp — Precision Tiers, GGUF Conversion & K-Quants

In this companion notebook, you will:
1. Install `llama-cpp-python` and `huggingface_hub` on Google Colab.
2. Download a compact open-weight **GGUF** checkpoint (`Qwen/Qwen2.5-0.5B-Instruct-GGUF`, `q4_k_m`).
3. Run CPU/GPU inference and compare memory footprint and generation speed.

In [ ]:
!pip install -q -U llama-cpp-python huggingface_hub

## 1. Download & Run a `Q4_K_M` GGUF Model with `llama-cpp-python`
The `Q4_K_M` K-quant format packs weights into 256-element super-blocks, balancing ~4.85 bits/weight with near-FP16 reasoning quality.

In [ ]:
import os
import time
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

gguf_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
    filename="qwen2.5-0.5b-instruct-q4_k_m.gguf"
)
size_mb = os.path.getsize(gguf_path) / (1024 ** 2)
print(f"Downloaded GGUF ({size_mb:.1f} MB): {gguf_path}")

llm = Llama(model_path=gguf_path, n_ctx=2048, n_threads=2, verbose=False)
t0 = time.perf_counter()
out = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Explain GGUF K-quants in two concise sentences."}],
    max_tokens=80,
    temperature=0.2,
)
dt = time.perf_counter() - t0
print(f"Latency: {dt:.2f}s | Output:\n{out['choices'][0]['message']['content']}")